In [ ]:

from pathlib import Path
LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legaluit/LegalIR/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legaluit/LegalIR/selected-contexts")
BGE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/modeluit/bge-m3-kaggle")
JINA_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/modeluit/jina-colbert-v2-64")
OFFLINE_WHEEL_DIR = Path("/kaggle/input/datasets/mduy2911/offline-packages")
BGE_MODEL_NAME, BGE_DECLARED_REVISION = "BAAI/bge-m3", "5617a9f61b028005a4858fdac845db406aefb181"
JINA_MODEL_NAME, JINA_DECLARED_REVISION = "jinaai/jina-colbert-v2-64", "7c9f323b61e6f96754300dea81bb51c3cc1d3f34"
MODEL_PARAMETER_LIMIT = 4_000_000_000
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS, EXPECTED_FIXED_CHUNKS = 8_532, 199_816
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K_CHUNKS, FINAL_K = 2_000, 200, 2_000, 5
DENSE_MAX_LENGTH = 8_192
CORPUS_BATCH_SIZE, QUERY_BATCH_SIZE = 256, 64
COLBERT_DOCUMENT_BATCH_SIZE, COLBERT_QUERY_BATCH_SIZE = 64, 32
TOKEN_HITS_PER_QUERY_VECTOR, MAXSIM_BATCH_SIZE = 4_096, 256
EMBEDDING_DIMENSION = 64
BASELINE_TOLERANCE = 1e-3
DENSE_BASELINE = {"recall_at_10": 0.9227799227799228, "recall_at_20": 0.9497265122265123, "recall_at_50": 0.9769144144144144, "recall_at_100": 0.9819015444015444, "mrr": 0.7159583402955311}
RESULT_PATH = Path("/kaggle/working/jina_colbert_v2_64_retrieval_dev_results.json")


In [ ]:
import os, subprocess, sys
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

os.environ.update({"HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "CUDA_VISIBLE_DEVICES": "0"})
for path, label, directory in ((LEGALIR_SOURCE_PATH, "LegalIR source", False), (CORPUS_PATH, "corpus", True), (BGE_MODEL_PATH, "BGE snapshot", True), (JINA_MODEL_PATH, "Jina snapshot", True), (OFFLINE_WHEEL_DIR, "offline wheel directory", True)):
    if not (path.is_dir() if directory else path.is_file()):
        raise FileNotFoundError(f"Attach {label} at {path}")

requirements = []
try:
    if Version(version("sentence-transformers")) < Version("6.0.0"):
        requirements.append("sentence-transformers>=6.0.0")
except PackageNotFoundError:
    requirements.append("sentence-transformers>=6.0.0")
try:
    version("einops")
except PackageNotFoundError:
    requirements.append("einops")
if requirements:
    wheels = sorted(OFFLINE_WHEEL_DIR.glob("*.whl"))
    if not wheels:
        raise FileNotFoundError(f"Attach offline wheels for {requirements} under {OFFLINE_WHEEL_DIR}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(OFFLINE_WHEEL_DIR), *requirements])
try:
    import faiss
except ImportError as exc:
    raise RuntimeError("Jina retrieval requires a Kaggle-compatible FAISS GPU installation or attached offline wheel") from exc
if not all(hasattr(faiss, name) for name in ("StandardGpuResources", "GpuIndexFlatConfig", "GpuIndexFlatIP")):
    raise RuntimeError("The installed FAISS build lacks required GPU flat-index APIs")


In [ ]:
# Standalone fixed-DEV LegalIR implementation. No repository runtime is imported.
import gc
import json
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from statistics import median
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_dev(path: Path) -> tuple[dict, dict]:
    source_digest = sha256(path.read_bytes()).hexdigest()
    if source_digest != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            f"LegalIR source SHA-256 mismatch: expected {EXPECTED_SOURCE_SHA256}, "
            f"got {source_digest}. Stop before reading samples."
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(isinstance(v, dict) for v in value.values()):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError("duplicate sample IDs after string canonicalization")

    split_counts = {"train": 0, "dev": 0, "holdout": 0}
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = question if isinstance(question, str) else f"\0fallback-sample-id:{sample_id}"
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        partition = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        split_counts[partition] += 1
        if partition == "dev":
            dev_ids.append(sample_id)
    if split_counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(f"fixed split counts mismatch: {split_counts}; stop")

    dev_ids.sort()
    dev = {sample_id: samples[sample_id] for sample_id in dev_ids}
    del samples, value
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        answer = sample.get("answer")
        if not isinstance(answer, list) or not answer:
            raise ValueError(f"DEV sample {sample_id!r}: expected a non-empty answer list")
        canonical = [str(document_id) for document_id in answer]
        if len(canonical) != len(set(canonical)):
            raise ValueError(f"DEV sample {sample_id!r}: answer contains duplicate IDs")
    evaluated_partition = "dev"
    if evaluated_partition != "dev" or len(dev) != EXPECTED_SPLIT_COUNTS["dev"]:
        raise RuntimeError("this notebook may evaluate fixed DEV only")
    return dev, {
        "version": "legalir_split_v1",
        "evaluated_partition": evaluated_partition,
        "queries": len(dev),
        "source_sha256": source_digest,
        "split_counts": split_counts,
        "selection_only": True,
        "holdout_metrics_computed": False,
    }


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json")
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = []
    for document in documents:
        if document.get("id") is None:
            raise ValueError("corpus document is missing a non-null ID")
        document_id = str(document["id"])
        if not isinstance(document.get("passage"), str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        document_ids.append(document_id)
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    if len(documents) != EXPECTED_DOCUMENTS:
        raise RuntimeError(f"expected {EXPECTED_DOCUMENTS} documents, got {len(documents)}")
    return documents


def distribution(values: list[int]) -> dict:
    array = np.asarray(values, dtype=np.int64)
    if array.size == 0:
        return {"min": None, "median": None, "p95": None, "max": None}
    return {
        "min": int(array.min()),
        "median": float(np.median(array)),
        "p95": float(np.percentile(array, 95)),
        "max": int(array.max()),
    }


def validate_chunk_provenance(documents: list[dict], chunks: list[dict]) -> dict:
    source_by_id = {str(document["id"]): document["passage"] for document in documents}
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("chunk IDs must be unique")
    intervals = defaultdict(list)
    for chunk in chunks:
        document_id = chunk["document_id"]
        source = source_by_id.get(document_id)
        if source is None:
            raise RuntimeError("chunk references an unknown document")
        start, end = chunk["char_start"], chunk["char_end"]
        if not (0 <= start < end <= len(source)) or chunk["text"] != source[start:end]:
            raise RuntimeError(f"chunk {chunk['chunk_id']!r}: exact provenance failed")
        intervals[document_id].append((start, end))
    for document_id, source in source_by_id.items():
        if not source:
            continue
        ordered = sorted(intervals[document_id])
        if not ordered or ordered[0][0] != 0:
            raise RuntimeError(f"document {document_id!r}: coverage does not start at zero")
        covered_end = 0
        for start, end in ordered:
            if start > covered_end:
                raise RuntimeError(f"document {document_id!r}: chunk coverage gap")
            covered_end = max(covered_end, end)
        if covered_end != len(source):
            raise RuntimeError(f"document {document_id!r}: incomplete source coverage")
    return {
        "exact_source_slices": True,
        "unique_chunk_ids": True,
        "complete_non_empty_source_coverage": True,
    }


def fixed_window_chunks(documents: list[dict], chunk_size: int, overlap: int) -> tuple[list[dict], dict]:
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("invalid fixed-window parameters")
    step = chunk_size - overlap
    chunks = []
    counts = []
    for document in documents:
        document_id = str(document["id"])
        source = document["passage"]
        before = len(chunks)
        for chunk_index, start in enumerate(range(0, len(source), step)):
            end = min(start + chunk_size, len(source))
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "chunk_index": chunk_index,
                "text": source[start:end],
                "char_start": start,
                "char_end": end,
            })
            if end == len(source):
                break
        counts.append(len(chunks) - before)
    provenance = validate_chunk_provenance(documents, chunks)
    return chunks, {
        "number_of_chunks": len(chunks),
        "chunk_size": chunk_size,
        "overlap": overlap,
        "step": step,
        "chunk_length_characters": distribution([len(chunk["text"]) for chunk in chunks]),
        "chunks_per_document": distribution(counts),
        "provenance": provenance,
    }


def model_metadata(
    model, model_name: str, declared_revision: str, path: Path,
    trust_remote_code: bool = False,
) -> dict:
    import re
    if not re.fullmatch(r"[0-9a-f]{40}", declared_revision):
        raise RuntimeError(
            f"{model_name}: replace declared revision with the resolved 40-hex snapshot commit SHA"
        )
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    eligible = parameter_count < MODEL_PARAMETER_LIMIT
    print(
        "Model eligibility:\n"
        f"model = {model_name}\n"
        f"parameter_count = {parameter_count}\n"
        f"competition_limit = < {MODEL_PARAMETER_LIMIT:,}\n"
        f"eligible = {str(eligible).lower()}"
    )
    if not eligible:
        raise RuntimeError(f"{model_name} is ineligible: parameter_count >= 4,000,000,000")
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is not None and config_hash != declared_revision:
        raise RuntimeError(
            f"{model_name} config revision {config_hash!r} != declared {declared_revision!r}"
        )
    return {
        "model_repository": model_name,
        "declared_revision": declared_revision,
        "resolved_snapshot": config_hash or declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": "verified-from-config" if config_hash else "declared-offline-snapshot",
        "actual_parameter_count": int(parameter_count),
        "competition_parameter_limit_exclusive": MODEL_PARAMETER_LIMIT,
        "eligible_under_4b_rule": eligible,
        "local_path": str(path),
        "local_files_only": True,
        "trust_remote_code": trust_remote_code,
    }



def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings = []
    started = perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch, padding=True, truncation=True, max_length=DENSE_MAX_LENGTH, return_tensors="pt"
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            output = model_bundle["model"](**inputs, return_dict=True)
            embedding = F.normalize(output.last_hidden_state[:, 0], p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def retrieve_dense_hits(query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor, chunks: list[dict], sample_ids: list[str]) -> dict:
    if query_embeddings.shape[0] != len(sample_ids) or corpus_embeddings.shape[0] != len(chunks):
        raise ValueError("embedding count mismatch")
    if len(chunks) < TOP_K_CHUNKS:
        raise ValueError("corpus has fewer chunks than requested retrieval depth")
    started = perf_counter()
    corpus_gpu = corpus_embeddings.to("cuda")
    hits = {}
    for start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[start:start + QUERY_BATCH_SIZE]
        query_gpu = query_embeddings[start:start + len(batch_ids)].to("cuda")
        similarities = query_gpu @ corpus_gpu.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense similarity contains non-finite values")
        scores, indices = torch.topk(similarities, k=TOP_K_CHUNKS, dim=1, sorted=True)
        for row, sample_id in enumerate(batch_ids):
            ordered = list(zip(scores[row].float().cpu().tolist(), indices[row].cpu().tolist()))
            ordered.sort(key=lambda item: (-item[0], item[1]))
            hits[sample_id] = [
                {
                    "chunk_index": int(chunk_index),
                    "document_id": chunks[int(chunk_index)]["document_id"],
                    "score": float(score),
                    "chunk_rank": rank,
                }
                for rank, (score, chunk_index) in enumerate(ordered, start=1)
            ]
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_gpu
    torch.cuda.empty_cache()
    return {"hits": hits, "seconds": seconds}


def dense_aggregate(scores: list[float], rule: str) -> float:
    ordered = sorted(scores, reverse=True)
    if rule == "max_top1":
        return ordered[0]
    if rule == "sum_top2":
        return sum(ordered[:2])
    if rule == "mean_top2":
        selected = ordered[:2]
        return sum(selected) / len(selected)
    if rule == "sum_top3":
        return sum(ordered[:3])
    raise ValueError(f"unknown dense aggregation rule: {rule}")


def candidates_from_hits(
    hits_by_query: dict,
    rule: str,
    depth: int,
    support_limit: int = 8,
    require_exact_depth: bool = True,
) -> dict:
    output = {}
    for sample_id, hits in hits_by_query.items():
        grouped = defaultdict(list)
        for hit in hits:
            if not isfinite(hit["score"]):
                raise RuntimeError("non-finite dense hit")
            grouped[hit["document_id"]].append(hit)
        documents = []
        for document_id, document_hits in grouped.items():
            ordered = sorted(document_hits, key=lambda item: (-item["score"], item["chunk_rank"], item["chunk_index"]))
            documents.append({
                "document_id": document_id,
                "dense_document_score": dense_aggregate([item["score"] for item in ordered], rule),
                "best_chunk_rank": ordered[0]["chunk_rank"],
                "available_global_hits": len(ordered),
                "supporting_chunk_indices": [item["chunk_index"] for item in ordered[:support_limit]],
            })
        documents.sort(key=lambda item: (-item["dense_document_score"], item["best_chunk_rank"], item["document_id"]))
        selected = documents[:depth]
        if not selected:
            raise RuntimeError(f"sample {sample_id!r}: dense retrieval produced no candidates")
        if require_exact_depth and len(selected) != depth:
            raise RuntimeError(f"sample {sample_id!r}: expected {depth} candidates")
        for rank, document in enumerate(selected, start=1):
            document["original_dense_rank"] = rank
        ids = [document["document_id"] for document in selected]
        if len(ids) != len(set(ids)):
            raise RuntimeError("candidate ranking contains duplicate document IDs")
        output[sample_id] = selected
    return output


def rankings_from_candidates(candidates: dict) -> dict:
    return {
        sample_id: [document["document_id"] for document in documents]
        for sample_id, documents in candidates.items()
    }


def evaluate_rankings(samples: dict, rankings: dict, depths=(5, 10, 20, 50, 100)) -> dict:
    if set(samples) != set(rankings):
        raise RuntimeError("ranking IDs do not match fixed DEV IDs")
    recalls = {depth: [] for depth in depths}
    reciprocal_ranks = []
    precision_5, recall_5 = [], []
    for sample_id, sample in samples.items():
        ranked = [str(document_id) for document_id in rankings[sample_id]]
        if len(ranked) != len(set(ranked)):
            raise RuntimeError(f"sample {sample_id!r}: duplicate ranked document IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth in depths:
            effective_k = min(depth, len(ranked))
            recalls[depth].append(len(gold.intersection(ranked[:effective_k])) / len(gold))
        first = next((rank for rank, document_id in enumerate(ranked, 1) if document_id in gold), None)
        reciprocal_ranks.append(0.0 if first is None else 1.0 / first)
        predicted = ranked[:FINAL_K]
        if not 1 <= len(predicted) <= 5:
            raise RuntimeError("prediction length must be between one and five")
        overlap = len(gold.intersection(predicted))
        precision_5.append(overlap / len(predicted))
        recall_5.append(overlap / len(gold))
    return {
        "precision": float(np.mean(precision_5)),
        "recall": float(np.mean(recall_5)),
        "mrr": float(np.mean(reciprocal_ranks)),
        **{f"recall_at_{depth}": float(np.mean(values)) for depth, values in recalls.items()},
    }


def first_gold_bins(samples: dict, rankings: dict) -> dict:
    bins = Counter()
    found = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        first = next((rank for rank, document_id in enumerate(rankings[sample_id], 1) if document_id in gold), None)
        if first is None:
            bins["not_found"] += 1
            continue
        found.append(first)
        label = "1" if first == 1 else "2_5" if first <= 5 else "6_10" if first <= 10 else "11_20" if first <= 20 else "21_50" if first <= 50 else "51_100" if first <= 100 else "beyond_100"
        bins[label] += 1
    labels = ("1", "2_5", "6_10", "11_20", "21_50", "51_100", "beyond_100", "not_found")
    return {
        "counts": {label: bins[label] for label in labels},
        "median_when_found": float(median(found)) if found else None,
    }


def metric_delta(alternative: dict, control: dict) -> dict:
    return {key: alternative[key] - control[key] for key in control if isinstance(control[key], float)}


def assert_dense_baseline(metrics: dict) -> None:
    observed = {key: metrics[key] for key in DENSE_BASELINE}
    delta = {key: observed[key] - DENSE_BASELINE[key] for key in DENSE_BASELINE}
    if max(abs(value) for value in delta.values()) > BASELINE_TOLERANCE:
        raise RuntimeError(f"fixed-window dense control mismatch: {delta}; stop before variants")


def save_result(result: dict) -> None:
    if result["split"]["evaluated_partition"] != "dev":
        raise RuntimeError("refusing to write a non-DEV result")
    RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("Saved aggregate-only DEV result:", RESULT_PATH)


def assert_local_remote_code_snapshot(path: Path) -> dict:
    inspected = []
    remote_references = []

    def walk(value):
        if isinstance(value, dict):
            for item in value.values():
                walk(item)
        elif isinstance(value, list):
            for item in value:
                walk(item)
        elif isinstance(value, str) and "--" in value:
            remote_references.append(value)

    for filename in ("config.json", "tokenizer_config.json", "modules.json"):
        candidate = path / filename
        if candidate.is_file():
            inspected.append(filename)
            walk(json.loads(candidate.read_text(encoding="utf-8")))
    if remote_references:
        raise RuntimeError(
            "Snapshot still contains cross-repository remote-code references: "
            f"{sorted(set(remote_references))}. Vendor the referenced Python files and rewrite auto_map locally."
        )
    python_files = sorted(str(item.relative_to(path)) for item in path.rglob("*.py"))
    if not python_files:
        raise RuntimeError(
            f"{path}: trust_remote_code=True requires vendored local Python implementation files"
        )
    return {
        "inspected_metadata_files": inspected,
        "vendored_python_files": python_files,
        "cross_repository_auto_map_references": [],
        "offline_remote_code_ready": True,
    }


def load_encoder(path: Path, name: str, revision: str, trust_remote_code: bool) -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True, trust_remote_code=trust_remote_code)
    model = AutoModel.from_pretrained(path, dtype=torch.float16, local_files_only=True, trust_remote_code=trust_remote_code)
    if int(getattr(model.config, "max_position_embeddings", 0)) < DENSE_MAX_LENGTH:
        raise RuntimeError(f"{name} does not support max_length=8192")
    metadata = model_metadata(model, name, revision, path, trust_remote_code)
    model.to("cuda").eval()
    return {"tokenizer": tokenizer, "model": model, "metadata": metadata, "load_seconds": perf_counter() - started}


def complementarity_at(samples: dict, a: dict, b: dict, depth: int) -> dict:
    query_counts, gold_counts = Counter(), Counter()
    for sid, sample in samples.items():
        gold = {str(x) for x in sample["answer"]}; sa, sb = set(a[sid][:depth]), set(b[sid][:depth])
        qa, qb = bool(gold & sa), bool(gold & sb)
        query_counts["both" if qa and qb else "BGE_only" if qa else "challenger_only" if qb else "neither"] += 1
        for doc in gold:
            ia, ib = doc in sa, doc in sb
            gold_counts["both" if ia and ib else "BGE_only" if ia else "challenger_only" if ib else "neither"] += 1
    return {"depth": depth, "query_coverage": dict(query_counts), "gold_document_coverage": dict(gold_counts)}


def rank_overlap_and_correlation(a: dict, b: dict, depth: int) -> dict:
    overlaps, correlations = [], []
    for sid in a:
        aa, bb = a[sid][:depth], b[sid][:depth]
        shared = sorted(set(aa) & set(bb)); overlaps.append(len(shared) / depth)
        if len(shared) >= 2:
            ra = np.asarray([aa.index(doc) + 1 for doc in shared], dtype=np.float64)
            rb = np.asarray([bb.index(doc) + 1 for doc in shared], dtype=np.float64)
            correlations.append(float(np.corrcoef(ra, rb)[0, 1]))
    return {"depth": depth, "mean_overlap_fraction": float(np.mean(overlaps)), "mean_shared_document_rank_correlation": float(np.nanmean(correlations)), "queries_with_correlation": len(correlations)}



def load_bge():
    tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(BGE_MODEL_PATH, dtype=torch.float16, local_files_only=True)
    metadata = model_metadata(model, BGE_MODEL_NAME, BGE_DECLARED_REVISION, BGE_MODEL_PATH, False)
    model.to("cuda").eval(); return {"tokenizer": tokenizer, "model": model, "metadata": metadata}


def load_colbert():
    import re
    from sentence_transformers import MultiVectorEncoder
    started = perf_counter()
    model = MultiVectorEncoder(str(JINA_MODEL_PATH), device="cuda", trust_remote_code=True, local_files_only=True, model_kwargs={"torch_dtype": torch.float16})
    parameter_count = sum(parameter.numel() for parameter in model.parameters()); eligible = parameter_count < MODEL_PARAMETER_LIMIT
    print(f"Model eligibility:\nmodel = {JINA_MODEL_NAME}\nparameter_count = {parameter_count}\ncompetition_limit = < {MODEL_PARAMETER_LIMIT:,}\neligible = {str(eligible).lower()}")
    if not eligible: raise RuntimeError("Jina ColBERT is ineligible under the <4B rule")
    if not re.fullmatch(r"[0-9a-f]{40}", JINA_DECLARED_REVISION): raise RuntimeError("Replace JINA_DECLARED_REVISION with the resolved snapshot commit SHA")
    return {"model": model, "metadata": {"model_repository": JINA_MODEL_NAME, "declared_revision": JINA_DECLARED_REVISION, "resolved_snapshot": JINA_DECLARED_REVISION, "actual_parameter_count": int(parameter_count), "competition_parameter_limit_exclusive": MODEL_PARAMETER_LIMIT, "eligible_under_4b_rule": eligible, "local_path": str(JINA_MODEL_PATH), "local_files_only": True, "trust_remote_code": True}, "load_seconds": perf_counter() - started}


def exact_maxsim(query_embedding: torch.Tensor, document_embeddings: list[np.ndarray]) -> torch.Tensor:
    query = query_embedding.to("cuda", dtype=torch.float32)
    output = []
    for start in range(0, len(document_embeddings), MAXSIM_BATCH_SIZE):
        batch = document_embeddings[start:start + MAXSIM_BATCH_SIZE]
        max_tokens = max(item.shape[0] for item in batch)
        padded = torch.zeros((len(batch), max_tokens, EMBEDDING_DIMENSION), device="cuda", dtype=torch.float32)
        mask = torch.zeros((len(batch), max_tokens), device="cuda", dtype=torch.bool)
        for row, item in enumerate(batch):
            tensor = torch.from_numpy(item.astype(np.float32, copy=False)).to("cuda")
            padded[row, :tensor.shape[0]] = tensor; mask[row, :tensor.shape[0]] = True
        similarities = torch.einsum("qd,bkd->bqk", query, padded).masked_fill(~mask[:, None, :], -torch.inf)
        output.append(similarities.max(dim=2).values.sum(dim=1).cpu())
    return torch.cat(output)


def build_colbert_index(model, texts: list[str]):
    import faiss
    resources = faiss.StandardGpuResources(); config = faiss.GpuIndexFlatConfig(); config.device = 0; config.useFloat16 = True
    index = faiss.GpuIndexFlatIP(resources, EMBEDDING_DIMENSION, config)
    embeddings, token_to_chunk, token_count = [], [], 0
    started = perf_counter()
    for start in range(0, len(texts), COLBERT_DOCUMENT_BATCH_SIZE):
        encoded = model.encode_document(texts[start:start + COLBERT_DOCUMENT_BATCH_SIZE], batch_size=COLBERT_DOCUMENT_BATCH_SIZE, show_progress_bar=False, convert_to_numpy=False)
        for offset, value in enumerate(encoded):
            array = value.detach().cpu().float().numpy().astype(np.float16)
            if array.ndim != 2 or array.shape[1] != EMBEDDING_DIMENSION or not np.isfinite(array).all(): raise RuntimeError("invalid ColBERT document token embeddings")
            embeddings.append(array); token_to_chunk.append(np.full(array.shape[0], start + offset, dtype=np.int32)); index.add(array.astype(np.float32)); token_count += array.shape[0]
    mapping = np.concatenate(token_to_chunk)
    return {"index": index, "gpu_resources": resources, "embeddings": embeddings, "token_to_chunk": mapping, "tokens": token_count, "seconds": perf_counter() - started, "embedding_bytes": int(sum(item.nbytes for item in embeddings)), "mapping_bytes": int(mapping.nbytes)}


def retrieve_colbert(model, index_bundle, questions: list[str], sample_ids: list[str], chunks: list[dict]):
    started = perf_counter(); hits = {}; candidate_counts = []
    query_embeddings = model.encode_query(questions, batch_size=COLBERT_QUERY_BATCH_SIZE, show_progress_bar=False, convert_to_numpy=False)
    for sid, query in zip(sample_ids, query_embeddings):
        q = query.detach().cpu().float()
        _, token_ids = index_bundle["index"].search(q.numpy(), TOKEN_HITS_PER_QUERY_VECTOR)
        candidate_indices = sorted(set(int(index_bundle["token_to_chunk"][idx]) for idx in token_ids.reshape(-1) if idx >= 0))
        if len(candidate_indices) < TOP_K_CHUNKS: raise RuntimeError("token ANN produced fewer than 2000 candidate chunks")
        candidate_counts.append(len(candidate_indices))
        candidate_embeddings = [index_bundle["embeddings"][idx] for idx in candidate_indices]
        scores = exact_maxsim(q, candidate_embeddings).tolist()
        ordered = sorted(zip(scores, candidate_indices), key=lambda item: (-item[0], item[1]))[:TOP_K_CHUNKS]
        hits[sid] = [{"chunk_index": idx, "document_id": chunks[idx]["document_id"], "score": float(score), "chunk_rank": rank} for rank, (score, idx) in enumerate(ordered, 1)]
    return {"hits": hits, "seconds": perf_counter() - started, "ann_candidate_chunks": distribution(candidate_counts), "query_embeddings": query_embeddings}


In [ ]:

run_started = perf_counter(); remote_code = assert_local_remote_code_snapshot(JINA_MODEL_PATH)
dev, split_info = load_fixed_dev(LEGALIR_SOURCE_PATH); documents = load_corpus(CORPUS_PATH); chunks, chunking = fixed_window_chunks(documents, CHUNK_SIZE, CHUNK_OVERLAP)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_FIXED_CHUNKS: raise RuntimeError("fixed corpus mismatch")
texts, questions, sample_ids = [x["text"] for x in chunks], [x["question"] for x in dev.values()], list(dev)
bge = load_bge(); bge_metadata = dict(bge["metadata"]); bge_corpus = encode_normalized_cls(bge, texts, CORPUS_BATCH_SIZE); bge_queries = encode_normalized_cls(bge, questions, QUERY_BATCH_SIZE)
bge_retrieval = retrieve_dense_hits(bge_queries["embeddings"], bge_corpus["embeddings"], chunks, sample_ids); bge_candidates = candidates_from_hits(bge_retrieval["hits"], "sum_top2", 200); bge_rankings = rankings_from_candidates(bge_candidates); bge_metrics = evaluate_rankings(dev, bge_rankings, depths=(10, 20, 50, 100, 200)); assert_dense_baseline(bge_metrics)
bge["model"].to("cpu"); del bge, bge_corpus, bge_queries, bge_candidates; gc.collect(); torch.cuda.empty_cache()
colbert = load_colbert(); colbert_model = colbert["model"]; index_bundle = build_colbert_index(colbert_model, texts)
colbert_retrieval = retrieve_colbert(colbert_model, index_bundle, questions, sample_ids, chunks); colbert_candidates = candidates_from_hits(colbert_retrieval["hits"], "sum_top2", 200); colbert_rankings = rankings_from_candidates(colbert_candidates); colbert_metrics = evaluate_rankings(dev, colbert_rankings, depths=(10, 20, 50, 100, 200))
result = {
    "experiment_name": "jina_colbert_v2_64_retrieval_dev", "split": split_info, "source_sha256": EXPECTED_SOURCE_SHA256, "query_count": len(dev),
    "research_question": "Does multilingual late interaction recover relevant documents missed by BGE-M3?", "research_axis": "single-vector dense versus ColBERT token-level late interaction", "control": BGE_MODEL_NAME, "independent_variable": JINA_MODEL_NAME,
    "fixed_components": {"chunking": "2000/200", "top_k_chunks": 2000, "document_aggregation": "sum top2", "matryoshka_optimization": False, "cross_encoder": None},
    "models": {"bge": bge_metadata, "jina_colbert": colbert["metadata"]}, "jina_offline_remote_code": remote_code,
    "late_interaction": {"query_encoder": "MultiVectorEncoder.encode_query", "document_encoder": "MultiVectorEncoder.encode_document", "query_document_markers": "model-configured asymmetric prompts", "score": "exact MaxSim sum over query tokens after token-ANN chunk candidate generation", "pooled_vector_used": False, "embedding_dimension": EMBEDDING_DIMENSION, "token_hits_per_query_vector": TOKEN_HITS_PER_QUERY_VECTOR},
    "chunking": chunking, "retrieval_metrics": {"bge": bge_metrics, "jina_colbert": colbert_metrics},
    "gold_complementarity": {str(depth): complementarity_at(dev, bge_rankings, colbert_rankings, depth) for depth in (100, 200)},
    "top_k_overlap": {str(depth): rank_overlap_and_correlation(bge_rankings, colbert_rankings, depth) for depth in (20, 50, 100)},
    "index_and_embedding_size": {"faiss_token_index_vectors": index_bundle["tokens"], "faiss_gpu_index_storage_estimate_bytes_fp16": index_bundle["tokens"] * EMBEDDING_DIMENSION * 2, "document_token_count": index_bundle["tokens"], "document_embedding_bytes_fp16": index_bundle["embedding_bytes"], "token_to_chunk_bytes": index_bundle["mapping_bytes"], "ann_candidate_chunks": colbert_retrieval["ann_candidate_chunks"]},
    "runtime": {"bge_retrieval_seconds": bge_retrieval["seconds"], "colbert_model_load_seconds": colbert["load_seconds"], "colbert_encoding_and_index_seconds": index_bundle["seconds"], "colbert_retrieval_seconds": colbert_retrieval["seconds"], "total_seconds": perf_counter() - run_started, "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated())},
    "interpretation": "Retrieval-only DEV evidence; no CE, holdout, public scoring, pooled-vector substitution, or automatic promotion."
}
save_result(result)
